# Notebook 8 — Context + Token RoBERTa

Subgroup identity is provided only inside `context_input_text`; there is no subgroup embedding and no FiLM.

In [9]:
import ast, json, random, itertools
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, confusion_matrix, classification_report
from scipy.spatial.distance import jensenshannon
from scipy.stats import entropy, pearsonr, spearmanr

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 220)

RANDOM_SEED = 42
MODEL_NAME = "roberta-base"
NUM_LABELS = 3

MAX_LENGTH = 384
BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 1

EPOCHS = 7
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
GRAD_CLIP = 1.0
DROPOUT = 0.1

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_DIR = Path("/home/shayan/Distributional-Hate-Speech-Prediction/data/processed")
CONTEXT_PATH = Path("/home/shayan/Distributional-Hate-Speech-Prediction/notebooks/context_artifacts/context_mapped_examples.parquet")
OUTPUT_DIR = Path("context_token_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_SEED)

print("Device:", DEVICE)
print("Context file:", CONTEXT_PATH)
print("Output directory:", OUTPUT_DIR.resolve())


Device: cuda
Context file: /home/shayan/Distributional-Hate-Speech-Prediction/notebooks/context_artifacts/context_mapped_examples.parquet
Output directory: /home/shayan/Distributional-Hate-Speech-Prediction/notebooks/context_token_outputs


## 1. Load data

In [10]:
context_df = pd.read_parquet(CONTEXT_PATH)

print("Context dataset:", context_df.shape)
display(context_df.head(2))

required_columns = [
    "comment_id", "split", "subgroup", "text", "target_distribution",
    "target_majority_label", "context_input_text", "retrieved_article_titles",
    "retrieved_summaries",
]
missing = [c for c in required_columns if c not in context_df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

print("Rows by split:")
display(context_df["split"].value_counts())

print("Rows by subgroup:")
display(context_df["subgroup"].value_counts())

print("Target majority distribution:")
display(context_df["target_majority_label"].value_counts(normalize=True).sort_index())

print("Unique context inputs:", context_df["context_input_text"].nunique(), "/", len(context_df))
print("Unique comments:", context_df["text"].nunique())

def safe_len(x):
    if isinstance(x, list): return len(x)
    if isinstance(x, np.ndarray): return len(x)
    if isinstance(x, str):
        try: return len(ast.literal_eval(x))
        except Exception: return np.nan
    return np.nan

print("Retrieved article count per row:")
display(context_df["retrieved_article_titles"].apply(safe_len).value_counts(dropna=False))

print("Example context:")
print("=" * 100)
print(context_df.iloc[0]["context_input_text"])


Context dataset: (9090, 16)


,experiment,comment_id,split,subgroup,subgroup_count,text,target_distribution,target_majority_label,target_expected_label,retrieved_article_titles,retrieved_page_urls,retrieved_similarities,retrieved_summaries,context_input_text,tweet_token_length,context_input_token_length
0,immigration,7,test,extremely_liberal,1,\*points to posters asking for palestinian rights and freedom\* theyre antisemetic and want to kill all the jews and dont acknowledge our right to exist ​ cant wait for this prick to die and go straight to hell. ill ...,"[1.0, 0.0, 0.0]",0,0.0,[Left-wing politics],[https://en.wikipedia.org/wiki/Left-wing_politics],[0.14819347858428955],"[Left-wing politics emphasizes social equality and egalitarianism, often opposing social hierarchies and advocating for the empowerment of marginalized groups. Leftist ideologies vary widely but typically involve a c...",### COMMENT TO CLASSIFY\n\*points to posters asking for palestinian rights and freedom\* theyre antisemetic and want to kill all the jews and dont acknowledge our right to exist ​ cant wait for this prick to die and ...,60,197
1,immigration,7,test,liberal,1,\*points to posters asking for palestinian rights and freedom\* theyre antisemetic and want to kill all the jews and dont acknowledge our right to exist ​ cant wait for this prick to die and go straight to hell. ill ...,"[0.0, 0.0, 1.0]",2,2.0,[Left-wing politics],[https://en.wikipedia.org/wiki/Left-wing_politics],[0.14819347858428955],"[Left-wing politics emphasizes social equality and egalitarianism, often opposing social hierarchies and advocating for the empowerment of marginalized groups. Leftist ideologies vary widely but typically involve a c...",### COMMENT TO CLASSIFY\n\*points to posters asking for palestinian rights and freedom\* theyre antisemetic and want to kill all the jews and dont acknowledge our right to exist ​ cant wait for this prick to die and ...,60,195


Rows by split:


split
train         6360
test          1384
validation    1346
Name: count, dtype: int64

Rows by subgroup:


...

Target majority distribution:


target_majority_label
0    0.734873
1    0.070957
2    0.194169
Name: proportion, dtype: float64

Unique context inputs: 9087 / 9090
Unique comments: 3797
Retrieved article count per row:


retrieved_article_titles
1    9090
Name: count, dtype: int64

Example context:
### COMMENT TO CLASSIFY
\*points to posters asking for palestinian rights and freedom\* theyre antisemetic and want to kill all the jews and dont acknowledge our right to exist ​ cant wait for this prick to die and go straight to hell. ill dig him the hole to get there myself

### ANNOTATOR IDEOLOGY
extremely_liberal

### RETRIEVED BACKGROUND
Left-wing politics:
Left-wing politics emphasizes social equality and egalitarianism, often opposing social hierarchies and advocating for the empowerment of marginalized groups. Leftist ideologies vary widely but typically involve a concern for those disadvantaged by society and a belief in reducing or abolishing unjustified inequalities through radical means. Key concepts include justice, solidarity, cultural pluralism, and freedom from forceful control or coercion. The left is associated with popular or state control of major institutions, trade unions, and critiques of capitalism, globalization, and environmental degradation.


## 2. Splits and tokenizer

In [11]:
train_df = context_df[context_df["split"] == "train"].reset_index(drop=True)
val_df = context_df[context_df["split"].isin(["validation", "val", "dev"])].reset_index(drop=True)
test_df = context_df[context_df["split"] == "test"].reset_index(drop=True)

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

assert len(train_df) > 0 and len(val_df) > 0 and len(test_df) > 0

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def count_tokens(text):
    return len(tokenizer(text, truncation=False, add_special_tokens=True)["input_ids"])

if "context_input_token_length" not in context_df.columns:
    context_df["context_input_token_length"] = context_df["context_input_text"].apply(count_tokens)

display(pd.DataFrame([{
    "mean": context_df["context_input_token_length"].mean(),
    "median": context_df["context_input_token_length"].median(),
    "p95": context_df["context_input_token_length"].quantile(0.95),
    "max": context_df["context_input_token_length"].max(),
    "pct_over_384": float((context_df["context_input_token_length"] > 384).mean()),
}]))


Train: (6360, 16)
Val: (1346, 16)
Test: (1384, 16)


,mean,median,p95,max,pct_over_384
0,173.806051,168.0,225.0,328,0.0


## 3. Dataset and model

In [12]:
def parse_distribution(value):
    if isinstance(value, np.ndarray):
        return value.astype(float).tolist()
    if isinstance(value, list):
        return [float(x) for x in value]
    if isinstance(value, str):
        return [float(x) for x in ast.literal_eval(value)]
    raise TypeError(f"Unsupported distribution type: {type(value)}")


class ContextTokenDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length):
        self.df = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        enc = self.tokenizer(
            row["context_input_text"],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "target_distribution": torch.tensor(parse_distribution(row["target_distribution"]), dtype=torch.float),
        }


class ContextTokenRoBERTa(nn.Module):
    def __init__(self, model_name, num_labels=3, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_labels),
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(cls)
        return {
            "logits": logits,
            "log_probs": torch.log_softmax(logits, dim=-1),
            "probs": torch.softmax(logits, dim=-1),
        }


train_loader = DataLoader(ContextTokenDataset(train_df, tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(ContextTokenDataset(val_df, tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(ContextTokenDataset(test_df, tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=False)

batch = next(iter(train_loader))
print({k: v.shape for k, v in batch.items()})

model = ContextTokenRoBERTa(MODEL_NAME, NUM_LABELS, DROPOUT).to(DEVICE)

criterion = nn.KLDivLoss(reduction="batchmean")
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

num_update_steps_per_epoch = int(np.ceil(len(train_loader) / GRADIENT_ACCUMULATION_STEPS))
num_training_steps = num_update_steps_per_epoch * EPOCHS
num_warmup_steps = int(WARMUP_RATIO * num_training_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps,
)

print("Training steps:", num_training_steps)


{'input_ids': torch.Size([8, 384]), 'attention_mask': torch.Size([8, 384]), 'target_distribution': torch.Size([8, 3])}


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Training steps: 5565


## 4. Metrics

In [13]:
EPS = 1e-12

def kl_divergence(y_true, y_pred):
    y_true = np.clip(y_true, EPS, 1.0)
    y_pred = np.clip(y_pred, EPS, 1.0)
    return np.sum(y_true * np.log(y_true / y_pred), axis=1)

def js_divergence(y_true, y_pred):
    return np.array([jensenshannon(t, p, base=2) ** 2 for t, p in zip(y_true, y_pred)])

def cross_entropy(y_true, y_pred):
    y_pred = np.clip(y_pred, EPS, 1.0)
    return -np.sum(y_true * np.log(y_pred), axis=1)

def expected_scores(distributions):
    return distributions @ np.arange(distributions.shape[1])

def entropy_values(distributions):
    return np.array([entropy(d, base=2) for d in distributions])

def compute_metrics(y_true, y_pred):
    true_labels = np.argmax(y_true, axis=1)
    pred_labels = np.argmax(y_pred, axis=1)
    true_entropy = entropy_values(y_true)
    pred_entropy = entropy_values(y_pred)

    out = {
        "kl_mean": float(kl_divergence(y_true, y_pred).mean()),
        "js_mean": float(js_divergence(y_true, y_pred).mean()),
        "cross_entropy_mean": float(cross_entropy(y_true, y_pred).mean()),
        "accuracy": float(accuracy_score(true_labels, pred_labels)),
        "macro_f1": float(f1_score(true_labels, pred_labels, average="macro", zero_division=0)),
        "expected_label_mae": float(mean_absolute_error(expected_scores(y_true), expected_scores(y_pred))),
    }
    if len(np.unique(true_entropy)) > 1 and len(np.unique(pred_entropy)) > 1:
        out["entropy_pearson"] = float(pearsonr(true_entropy, pred_entropy).statistic)
        out["entropy_spearman"] = float(spearmanr(true_entropy, pred_entropy).statistic)
    else:
        out["entropy_pearson"] = np.nan
        out["entropy_spearman"] = np.nan
    return out


## 5. Train

In [14]:
def run_epoch(model, dataloader, optimizer=None, scheduler=None):
    is_training = optimizer is not None
    model.train() if is_training else model.eval()

    total_loss = 0.0
    all_targets, all_predictions = [], []

    if is_training:
        optimizer.zero_grad()

    for step, batch in enumerate(dataloader):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        targets = batch["target_distribution"].to(DEVICE)

        with torch.set_grad_enabled(is_training):
            outputs = model(input_ids, attention_mask)
            raw_loss = criterion(outputs["log_probs"], targets)

            if is_training:
                (raw_loss / GRADIENT_ACCUMULATION_STEPS).backward()
                should_step = ((step + 1) % GRADIENT_ACCUMULATION_STEPS == 0) or ((step + 1) == len(dataloader))
                if should_step:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                    optimizer.step()
                    optimizer.zero_grad()
                    if scheduler is not None:
                        scheduler.step()

        total_loss += raw_loss.item() * input_ids.size(0)
        all_targets.append(targets.detach().cpu().numpy())
        all_predictions.append(outputs["probs"].detach().cpu().numpy())

    y_true = np.vstack(all_targets)
    y_pred = np.vstack(all_predictions)
    metrics = compute_metrics(y_true, y_pred)
    metrics["loss"] = float(total_loss / len(dataloader.dataset))
    return metrics, y_true, y_pred


best_val_kl = float("inf")
best_model_path = OUTPUT_DIR / "context_token_best_model.pt"
history = []

for epoch in range(1, EPOCHS + 1):
    train_metrics, _, _ = run_epoch(model, train_loader, optimizer, scheduler)
    val_metrics, _, _ = run_epoch(model, val_loader)

    row = {"epoch": epoch, **{f"train_{k}": v for k, v in train_metrics.items()}, **{f"val_{k}": v for k, v in val_metrics.items()}}
    history.append(row)

    print("=" * 80)
    print(f"Epoch {epoch}/{EPOCHS}")
    print(f"Train KL: {train_metrics['kl_mean']:.4f} | Val KL: {val_metrics['kl_mean']:.4f}")
    print(f"Train JS: {train_metrics['js_mean']:.4f} | Val JS: {val_metrics['js_mean']:.4f}")
    print(f"Val Acc: {val_metrics['accuracy']:.4f} | Val Macro F1: {val_metrics['macro_f1']:.4f}")

    if val_metrics["kl_mean"] < best_val_kl:
        best_val_kl = val_metrics["kl_mean"]
        torch.save(model.state_dict(), best_model_path)
        print("Saved best model")

history_df = pd.DataFrame(history)
display(history_df)
history_df.to_csv(OUTPUT_DIR / "context_token_training_history.csv", index=False)


Epoch 1/7
Train KL: 0.7157 | Val KL: 0.5996
Train JS: 0.2767 | Val JS: 0.2190
Val Acc: 0.7831 | Val Macro F1: 0.4639
Saved best model
Epoch 2/7
Train KL: 0.6064 | Val KL: 0.6089
Train JS: 0.2204 | Val JS: 0.2234
Val Acc: 0.7585 | Val Macro F1: 0.4603
Epoch 3/7
Train KL: 0.5525 | Val KL: 0.6686
Train JS: 0.1980 | Val JS: 0.2058
Val Acc: 0.7823 | Val Macro F1: 0.4755
Epoch 4/7
Train KL: 0.5112 | Val KL: 0.6082
Train JS: 0.1809 | Val JS: 0.2164
Val Acc: 0.7823 | Val Macro F1: 0.4680
Epoch 5/7
Train KL: 0.4677 | Val KL: 0.6818
Train JS: 0.1665 | Val JS: 0.2105
Val Acc: 0.7756 | Val Macro F1: 0.4566
Epoch 6/7
Train KL: 0.4353 | Val KL: 0.6925
Train JS: 0.1555 | Val JS: 0.2198
Val Acc: 0.7630 | Val Macro F1: 0.4558
Epoch 7/7
Train KL: 0.4006 | Val KL: 0.7432
Train JS: 0.1463 | Val JS: 0.2171
Val Acc: 0.7697 | Val Macro F1: 0.4631


,epoch,train_kl_mean,train_js_mean,train_cross_entropy_mean,train_accuracy,train_macro_f1,train_expected_label_mae,train_entropy_pearson,train_entropy_spearman,train_loss,val_kl_mean,val_js_mean,val_cross_entropy_mean,val_accuracy,val_macro_f1,val_expected_label_mae,val_entropy_pearson,val_entropy_spearman,val_loss
0,1,0.715714,0.276702,0.748165,0.734434,0.372236,0.625681,0.053267,0.059815,0.715714,0.599636,0.219049,0.633815,0.783061,0.463881,0.477900,0.194845,0.198265,0.599636
1,2,0.606429,0.220365,0.638880,0.772799,0.466930,0.476191,0.138557,0.132439,0.606429,0.608895,0.223437,0.643073,0.758544,0.460251,0.491964,0.145061,0.134471,0.608895
2,3,0.552467,0.198032,0.584918,0.794340,0.493494,0.413366,0.153472,0.141313,0.552467,0.668642,0.205835,0.702820,0.782318,0.475487,0.417584,0.158326,0.156606,0.668642
3,4,0.511249,0.180929,0.543700,0.806918,0.508164,0.366521,0.162939,0.151762,0.511249,0.608215,0.216377,0.642393,0.782318,0.468025,0.455170,0.142971,0.135531,0.608215
4,5,0.467745,0.166511,0.500196,0.819025,0.522894,0.329942,0.178393,0.167118,0.467745,0.681821,0.210491,0.715999,0.775632,0.456617,0.422050,0.118777,0.122144,0.681821
5,6,0.435256,0.155537,0.467707,0.824843,0.529371,0.303422,0.179199,0.175534,0.435256,0.692471,0.219750,0.726649,0.763001,0.455810,0.443266,0.089149,0.099467,0.692471
6,7,0.400612,0.146315,0.433063,0.835220,0.541947,0.282899,0.206169,0.189955,0.400612,0.743199,0.217131,0.777377,0.769688,0.463105,0.429760,0.074547,0.109596,0.743199


## 6. Test and diagnostics

In [15]:
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
test_metrics, y_true_test, y_pred_test = run_epoch(model, test_loader)

display(test_metrics)

with open(OUTPUT_DIR / "context_token_test_metrics.json", "w") as f:
    json.dump(test_metrics, f, indent=2)

predictions_df = test_df.copy()
predictions_df["true_distribution"] = list(y_true_test)
predictions_df["pred_distribution"] = list(y_pred_test)
predictions_df["pred_majority_label"] = np.argmax(y_pred_test, axis=1)
predictions_df["pred_expected_label"] = expected_scores(y_pred_test)
predictions_df["pred_entropy"] = entropy_values(y_pred_test)
predictions_df["kl"] = kl_divergence(y_true_test, y_pred_test)
predictions_df["js"] = js_divergence(y_true_test, y_pred_test)
predictions_df["cross_entropy"] = cross_entropy(y_true_test, y_pred_test)

predictions_df.to_parquet(OUTPUT_DIR / "context_token_test_predictions.parquet", index=False)
predictions_df.to_csv(OUTPUT_DIR / "context_token_test_predictions.csv", index=False)

true_labels = np.argmax(y_true_test, axis=1)
pred_labels = np.argmax(y_pred_test, axis=1)

print("Confusion matrix:")
print(confusion_matrix(true_labels, pred_labels, labels=[0, 1, 2]))

print("\\nClassification report:")
print(classification_report(true_labels, pred_labels, labels=[0, 1, 2], zero_division=0))

print("\\nPredicted label distribution:")
display(pd.Series(pred_labels).value_counts(normalize=True).sort_index())

print("\\nTrue label distribution:")
display(pd.Series(true_labels).value_counts(normalize=True).sort_index())

print("\\nAverage predicted distribution:")
print(np.vstack(predictions_df["pred_distribution"].to_numpy()).mean(axis=0))

print("\\nMean KL by true majority label:")
display(
    predictions_df
    .groupby("target_majority_label")
    .agg(
        n=("comment_id", "count"),
        mean_kl=("kl", "mean"),
        mean_js=("js", "mean"),
        mean_target_entropy=("target_distribution", lambda x: np.mean([entropy(parse_distribution(v), base=2) for v in x])),
        mean_pred_entropy=("pred_entropy", "mean"),
    )
)

print("Average predicted distribution by true majority label:")
for label in [0, 1, 2]:
    subset = predictions_df[predictions_df["target_majority_label"] == label]
    avg_pred = np.vstack(subset["pred_distribution"].to_numpy()).mean(axis=0)
    print(label, avg_pred)

print("\\nPerformance by subgroup:")
subgroup_rows = []
for subgroup, group in predictions_df.groupby("subgroup"):
    y_true = np.vstack(group["true_distribution"].to_numpy())
    y_pred = np.vstack(group["pred_distribution"].to_numpy())
    subgroup_rows.append({"subgroup": subgroup, "n": len(group), **compute_metrics(y_true, y_pred)})

subgroup_metrics_df = pd.DataFrame(subgroup_rows).sort_values("kl_mean")
display(subgroup_metrics_df)
subgroup_metrics_df.to_csv(OUTPUT_DIR / "context_token_subgroup_metrics.csv", index=False)


/tmp/ipykernel_2362/1013887499.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))


{'kl_mean': 0.6163268089294434,
 'js_mean': 0.22943889575226015,
 'cross_entropy_mean': 0.6407516598701477,
 'accuracy': 0.7709537572254336,
 'macro_f1': 0.45647876254711295,
 'expected_label_mae': 0.5082833348898583,
 'entropy_pearson': 0.048189493289402544,
 'entropy_spearman': 0.04951723067570954,
 'loss': 0.616326829989177}

Confusion matrix:
[[943   0  86]
 [ 74   0  21]
 [136   0 124]]
\nClassification report:
              precision    recall  f1-score   support

           0       0.82      0.92      0.86      1029
           1       0.00      0.00      0.00        95
           2       0.54      0.48      0.51       260

    accuracy                           0.77      1384
   macro avg       0.45      0.46      0.46      1384
weighted avg       0.71      0.77      0.74      1384

\nPredicted label distribution:


0    0.833092
2    0.166908
Name: proportion, dtype: float64

\nTrue label distribution:


0    0.743497
1    0.068642
2    0.187861
Name: proportion, dtype: float64

\nAverage predicted distribution:
[0.7437989  0.06761107 0.18859026]
\nMean KL by true majority label:


,n,mean_kl,mean_js,mean_target_entropy,mean_pred_entropy
target_majority_label,,,,,
0,1029,0.268744,0.119015,0.035754,0.765186
1,95,2.590573,0.792561,0.052632,0.960007
2,260,1.270594,0.460706,0.026836,1.143103


Average predicted distribution by true majority label:
0 [0.80113715 0.06015013 0.13871269]
1 [0.68568313 0.07651816 0.23779881]
2 [0.5381062  0.09388439 0.36800936]
\nPerformance by subgroup:


,subgroup,n,kl_mean,js_mean,cross_entropy_mean,accuracy,macro_f1,expected_label_mae,entropy_pearson,entropy_spearman
1,extremely_conservative,46,0.442295,0.180601,0.452841,0.826087,0.518519,0.436768,0.195539,0.151587
0,conservative,163,0.505838,0.193125,0.517769,0.828221,0.502906,0.429775,0.002536,0.035236
6,slightly_conservative,154,0.533549,0.207403,0.545109,0.785714,0.467450,0.470491,-0.030536,-0.031671
5,no_opinion,46,0.543022,0.200228,0.561941,0.847826,0.557890,0.438510,-0.044538,-0.069980
3,liberal,308,0.603217,0.223633,0.643479,0.801948,0.482213,0.492948,0.046652,0.074876
4,neutral,257,0.678177,0.247430,0.711600,0.758755,0.445946,0.545546,0.092891,0.078375
7,slightly_liberal,213,0.687201,0.256158,0.704346,0.690141,0.379040,0.561381,0.075810,0.026250
2,extremely_liberal,197,0.693388,0.251654,0.714106,0.736041,0.429456,0.553732,0.026793,0.036163


## 7. Counterfactual subgroup sensitivity

In [16]:
@torch.no_grad()
def predict_distribution(context_text):
    model.eval()
    enc = tokenizer(
        context_text,
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )
    outputs = model(enc["input_ids"].to(DEVICE), enc["attention_mask"].to(DEVICE))
    return outputs["probs"].detach().cpu().numpy()[0]


subgroups = sorted(context_df["subgroup"].unique())
context_lookup = {(row["comment_id"], row["subgroup"]): row["context_input_text"] for _, row in context_df.iterrows()}
unique_test_comments = test_df[["comment_id", "text"]].drop_duplicates("comment_id").reset_index(drop=True)

counterfactual_rows = []
for _, row in unique_test_comments.iterrows():
    comment_id = row["comment_id"]
    available_subgroups = [s for s in subgroups if (comment_id, s) in context_lookup]

    if len(available_subgroups) < 2:
        continue

    preds_by_group = {s: predict_distribution(context_lookup[(comment_id, s)]) for s in available_subgroups}

    pairwise_js = []
    for group_a, group_b in itertools.combinations(available_subgroups, 2):
        pairwise_js.append(jensenshannon(preds_by_group[group_a], preds_by_group[group_b], base=2) ** 2)

    counterfactual_rows.append({
        "comment_id": comment_id,
        "text": row["text"],
        "n_available_subgroups": len(available_subgroups),
        "mean_pairwise_js": float(np.mean(pairwise_js)),
        "max_pairwise_js": float(np.max(pairwise_js)),
    })

cf_df = pd.DataFrame(counterfactual_rows)

print("Counterfactual rows:", cf_df.shape)
print("Mean counterfactual JS:", cf_df["mean_pairwise_js"].mean())
print("Median counterfactual JS:", cf_df["mean_pairwise_js"].median())
print("Mean max-pairwise JS:", cf_df["max_pairwise_js"].mean())
print("Median max-pairwise JS:", cf_df["max_pairwise_js"].median())

display(cf_df.sort_values("max_pairwise_js", ascending=False).head(30))
cf_df.to_csv(OUTPUT_DIR / "context_token_counterfactual_js.csv", index=False)


Counterfactual rows: (590, 5)
Mean counterfactual JS: 0.00021630985200130044
Median counterfactual JS: 1.4278380925233359e-05
Mean max-pairwise JS: 0.00029413424519196345
Median max-pairwise JS: 1.91161907518354e-05


,comment_id,text,n_available_subgroups,mean_pairwise_js,max_pairwise_js
540,48030,Omar and her ilk are traitors to the USA. Those who voted for this trash should be deported. Thanks voters for the TRUMP REVOLUTION!,3,0.010066,0.018298
386,41374,Screw Mexico we don't need them,3,0.006158,0.010817
171,20601,"Coming from the woman who chants convert or die. Sit down Linda. Better yet, deport yourself to the nearest sandy hole. URL",3,0.006143,0.009742
537,47857,"Rid this country of all illegals. Our veterans need the handout people, not them.",3,0.005053,0.008658
539,48024,Kick out illegals obviously,3,0.004562,0.007001
12,1940,Too bad australia isn't a country that goes around cutting off clits and shit that needs the change,2,0.005373,0.005373
384,41270,I like Japan you twit. Let me help you that sentence . WHITE American citizens don't give a crap about Japanese citizens abducted by North Korea. The Same White Americans who voted this Orange Shit Stain as our presi...,4,0.002257,0.004846
387,41440,Anthony Joseph Look up Procorp Israel perverts have ties everywhere. It's disgusting,2,0.004637,0.004637
161,19965,Unless they are from shit hole countries right?,2,0.004381,0.004381
240,29957,Stop making dumb Dominicans famous,3,0.001930,0.003828


## 8. EL vs EC and true-vs-model disagreement

In [17]:
def pairwise_counterfactual_js(group_a, group_b):
    rows = []
    unique_comments = test_df[["comment_id", "text"]].drop_duplicates("comment_id").reset_index(drop=True)

    for _, row in unique_comments.iterrows():
        comment_id = row["comment_id"]
        if (comment_id, group_a) not in context_lookup or (comment_id, group_b) not in context_lookup:
            continue

        context_a = context_lookup[(comment_id, group_a)]
        context_b = context_lookup[(comment_id, group_b)]

        pred_a = predict_distribution(context_a)
        pred_b = predict_distribution(context_b)

        rows.append({
            "comment_id": comment_id,
            "text": row["text"],
            "group_a": group_a,
            "group_b": group_b,
            "js": float(jensenshannon(pred_a, pred_b, base=2) ** 2),
            f"pred_{group_a}": pred_a,
            f"pred_{group_b}": pred_b,
            f"context_{group_a}": context_a,
            f"context_{group_b}": context_b,
        })

    return pd.DataFrame(rows)


def valid_dist(x):
    try:
        arr = np.array(parse_distribution(x), dtype=float)
    except Exception:
        return False
    return arr.shape[0] == NUM_LABELS and not np.isnan(arr).any() and arr.sum() > 0


def true_pairwise_disagreement_from_context_df(full_df, group_a, group_b):
    rows = []

    for comment_id, group in full_df.groupby("comment_id"):
        if group_a not in set(group["subgroup"]) or group_b not in set(group["subgroup"]):
            continue

        row_a = group[group["subgroup"] == group_a].iloc[0]
        row_b = group[group["subgroup"] == group_b].iloc[0]

        dist_a = parse_distribution(row_a["target_distribution"])
        dist_b = parse_distribution(row_b["target_distribution"])

        if not valid_dist(dist_a) or not valid_dist(dist_b):
            continue

        dist_a = np.array(dist_a, dtype=float)
        dist_b = np.array(dist_b, dtype=float)

        rows.append({
            "comment_id": comment_id,
            "text": row_a["text"],
            "true_js": float(jensenshannon(dist_a, dist_b, base=2) ** 2),
            f"{group_a}_true_dist": dist_a,
            f"{group_b}_true_dist": dist_b,
            f"{group_a}_true_label": int(np.argmax(dist_a)),
            f"{group_b}_true_label": int(np.argmax(dist_b)),
        })

    return pd.DataFrame(rows)


if "extremely_liberal" in subgroups and "extremely_conservative" in subgroups:
    extreme_df = pairwise_counterfactual_js("extremely_liberal", "extremely_conservative")

    print("EL vs EC rows:", extreme_df.shape)
    print("Extreme liberal vs extreme conservative mean JS:", extreme_df["js"].mean())
    print("Extreme liberal vs extreme conservative median JS:", extreme_df["js"].median())
    display(extreme_df.sort_values("js", ascending=False).head(30))
    extreme_df.to_csv(OUTPUT_DIR / "context_token_EL_vs_EC_counterfactual.csv", index=False)

    true_df = true_pairwise_disagreement_from_context_df(context_df, "extremely_liberal", "extremely_conservative")

    comparison_df = true_df.merge(
        extreme_df[["comment_id", "js", "pred_extremely_liberal", "pred_extremely_conservative"]],
        on="comment_id",
        how="inner",
    ).rename(columns={"js": "model_js"})

    nonzero = comparison_df[comparison_df["true_js"] > 0].copy()
    nonzero["capture_ratio"] = nonzero["model_js"] / nonzero["true_js"]

    print("N true EL/EC overlap:", len(true_df))
    print("N comparison:", len(comparison_df))
    print("Mean true JS:", comparison_df["true_js"].mean())
    print("Mean model JS:", comparison_df["model_js"].mean())
    if len(nonzero) > 0:
        print("Mean capture ratio, true_js > 0:", nonzero["capture_ratio"].mean())
        print("Median capture ratio, true_js > 0:", nonzero["capture_ratio"].median())

    comparison_df["label_pair"] = (
        comparison_df["extremely_liberal_true_label"].astype(str)
        + "-"
        + comparison_df["extremely_conservative_true_label"].astype(str)
    )

    display(
        comparison_df
        .groupby("label_pair")
        .agg(n=("comment_id", "count"), mean_true_js=("true_js", "mean"), mean_model_js=("model_js", "mean"))
        .sort_values("mean_true_js", ascending=False)
    )

    comparison_df.to_csv(OUTPUT_DIR / "context_token_true_vs_model_EL_EC.csv", index=False)

    if len(extreme_df) > 0:
        top_row = extreme_df.sort_values("js", ascending=False).iloc[0]
        print("\\nHighest EL/EC context effect")
        print("Comment:")
        print(top_row["text"])
        print("\\nPrediction EL:")
        print(top_row["pred_extremely_liberal"])
        print("\\nPrediction EC:")
        print(top_row["pred_extremely_conservative"])
        print("\\nContext EL:")
        print(top_row["context_extremely_liberal"])
        print("\\nContext EC:")
        print(top_row["context_extremely_conservative"])


EL vs EC rows: (8, 9)
Extreme liberal vs extreme conservative mean JS: 7.698935260527442e-05
Extreme liberal vs extreme conservative median JS: 1.1564636613895562e-06


,comment_id,text,group_a,group_b,js,pred_extremely_liberal,pred_extremely_conservative,context_extremely_liberal,context_extremely_conservative
4,37246,racists b like go back to ur country like ok stop bombing it n maybe i will,extremely_liberal,extremely_conservative,3.475054e-04,"[0.7484684, 0.082054414, 0.16947715]","[0.76687604, 0.07805495, 0.15506898]",### COMMENT TO CLASSIFY\nracists b like go back to ur country like ok stop bombing it n maybe i will\n\n### ANNOTATOR IDEOLOGY\nextremely_liberal\n\n### RETRIEVED BACKGROUND\nFree migration:\nThe concept of free migr...,### COMMENT TO CLASSIFY\nracists b like go back to ur country like ok stop bombing it n maybe i will\n\n### ANNOTATOR IDEOLOGY\nextremely_conservative\n\n### RETRIEVED BACKGROUND\nOpposition to immigration:\nNational...
2,20033,"STOP SUCKING THAT FATASS' DICK, YOU NIGGERLOVING KIKE",extremely_liberal,extremely_conservative,2.191093e-04,"[0.3122031, 0.11781855, 0.5699784]","[0.29626158, 0.11934488, 0.58439356]","### COMMENT TO CLASSIFY\nSTOP SUCKING THAT FATASS' DICK, YOU NIGGERLOVING KIKE\n\n### ANNOTATOR IDEOLOGY\nextremely_liberal\n\n### RETRIEVED BACKGROUND\nLeft-wing politics:\nLeft-wing politics emphasizes social equal...","### COMMENT TO CLASSIFY\nSTOP SUCKING THAT FATASS' DICK, YOU NIGGERLOVING KIKE\n\n### ANNOTATOR IDEOLOGY\nextremely_conservative\n\n### RETRIEVED BACKGROUND\nNational conservatism:\nNational conservatism prioritizes ..."
0,13695,Yes so every lazy poverty stricken uneducated third world citizen in the entire world from Africa to South America can come here and get welfare and turn America into just as bad of a shithole as their original count...,extremely_liberal,extremely_conservative,4.561319e-05,"[0.80951077, 0.068000086, 0.12248921]","[0.8033281, 0.0695985, 0.1270734]",### COMMENT TO CLASSIFY\nYes so every lazy poverty stricken uneducated third world citizen in the entire world from Africa to South America can come here and get welfare and turn America into just as bad of a shithol...,### COMMENT TO CLASSIFY\nYes so every lazy poverty stricken uneducated third world citizen in the entire world from Africa to South America can come here and get welfare and turn America into just as bad of a shithol...
5,38441,Indian cunts and dickheads orgasming over developing the occupied and annexed Kashmir so they can settle there while the Kashmiri natives remain locked down in their houses with the Indian occupation forces roaming t...,extremely_liberal,extremely_conservative,1.511377e-06,"[0.25910717, 0.121591136, 0.61930174]","[0.2579593, 0.122143485, 0.61989725]",### COMMENT TO CLASSIFY\nIndian cunts and dickheads orgasming over developing the occupied and annexed Kashmir so they can settle there while the Kashmiri natives remain locked down in their houses with the Indian oc...,### COMMENT TO CLASSIFY\nIndian cunts and dickheads orgasming over developing the occupied and annexed Kashmir so they can settle there while the Kashmiri natives remain locked down in their houses with the Indian oc...
3,20062,I've always been proud of the Igbo tribe and our culture. Yesterday made me even more proud. You people are too fine abeg. OgbakoUmuIgboDay,extremely_liberal,extremely_conservative,8.015500e-07,"[0.9057054, 0.043055665, 0.051238965]","[0.9063193, 0.042747382, 0.050933346]",### COMMENT TO CLASSIFY\nI've always been proud of the Igbo tribe and our culture. Yesterday made me even more proud. You people are too fine abeg. OgbakoUmuIgboDay\n\n### ANNOTATOR IDEOLOGY\nextremely_liberal\n\n###...,### COMMENT TO CLASSIFY\nI've always been proud of the Igbo tribe and our culture. Yesterday made me even more proud. You people are too fine abeg. OgbakoUmuIgboDay\n\n### ANNOTATOR IDEOLOGY\nextremely_conservative\n...
7,49332,Send all these asses back !,extremely_liberal,extremely_conservative,5.877474e-07,"[0.5286109, 0.10604784, 0.36534128]","[0.5278497, 0.105924666, 0.36622566]","### COMMENT TO CLASSIFY\nSend all these asses back !\n\n### ANNOTATOR IDE

N true EL/EC overlap: 101
N comparison: 8
Mean true JS: 0.16448066401642414
Mean model JS: 7.698935260527442e-05
Mean capture ratio, true_js > 0: 8.626808444947948e-05
Median capture ratio, true_js > 0: 0.00011168134001247791


,n,mean_true_js,mean_model_js
label_pair,,,
0-1,1,1.000000,5.877474e-07
0-0,4,0.077820,9.854909e-05
2-2,3,0.001522,7.371023e-05


\nHighest EL/EC context effect
Comment:
racists b like go back to ur country like ok stop bombing it n maybe i will
\nPrediction EL:
[0.7484684  0.08205441 0.16947715]
\nPrediction EC:
[0.76687604 0.07805495 0.15506898]
\nContext EL:
### COMMENT TO CLASSIFY
racists b like go back to ur country like ok stop bombing it n maybe i will

### ANNOTATOR IDEOLOGY
extremely_liberal

### RETRIEVED BACKGROUND
Free migration:
The concept of free migration has been debated globally, with various perspectives on its moral, economic, and cultural implications. Some argue that it's a human right, citing religious teachings such as Buddhism and Christianity, which emphasize hospitality and equality towards strangers. Others propose that open borders would lead to economic gains through increased labor mobility and reduced production efficiency gaps between countries. However, arguments against free migration often focus on economic, cultural, or security concerns, with some advocating for stricter immi

## Interpretation

This model answers whether putting subgroup identity and retrieved context directly through RoBERTa attention is enough, without subgroup embeddings or FiLM.

Compare against:

- Token baseline without context
- Embedding baseline without context
- Context + Embedding
- Context + Strong FiLM
